# Faruq-v3 AGSF synthesis screening

Validation-only, seed-42, fail-fast synthesis. `SYN0 = STB + ambiguity multilevel correction`; `SYN1 = SYN0 + additive AF2 cue`; `SYN2 = SYN0 + gated AF2 cue`. Native YOLO26 box branches consume untouched P3/P4/P5 features. Test is never extracted or opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/agsf-synthesis-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
command=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(command)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REQUIRED=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/STB1/val_reports/stb_seed42_screening.json',
)
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE=require_project_artifact(PROJECT_ROOT,REQUIRED[0])
D0=require_project_artifact(PROJECT_ROOT,REQUIRED[1])
STB_SUMMARY=require_project_artifact(PROJECT_ROOT,REQUIRED[2])
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
GROUPED=DATA_ROOT/'faruq_grouped_summary.json'
assert GROUPED.is_file() and not (DATA_ROOT/'test').exists()
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-agsf-synthesis-v1'
STATIC=OUTPUT/'static_audit.json'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0))
print('PROJECT:',PROJECT_ROOT)
print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.agsf.audit import static_agsf_audit
static=static_agsf_audit(REPO/'configs/coffee_fg/models/yolo26n-p3.yaml',D0,STATIC,nc=21,image_size=128,hidden_dim=64)
print('ARMS:',{name:{'decision':row['decision'],'parameters':row['parameters']} for name,row in static['arms'].items()})
print('CAPACITY:',static['capacity_gates'])
print('DECISION:',static['decision'])
assert static['decision']=='PASS','STOP: static wiring AGSF gagal.'


## Stage 1 — core synthesis
Melatih hanya SYN0. Jika SYN0 tidak mengalahkan STB1 sesuai gate, notebook berhenti dan AF2 tidak dilatih.

In [ ]:
base=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_agsf_synthesis','--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED),'--stb-summary',str(STB_SUMMARY),'--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
command=base+['--stage','core']
print('MENJALANKAN CORE:',' '.join(command),flush=True)
subprocess.run(command,cwd=REPO,check=True)


In [ ]:
import pandas as pd
from IPython.display import display
CORE=OUTPUT/'val_reports/agsf_core_seed42_decision.json'
core=json.loads(CORE.read_text())
rows=[{'model':'STB1',**core['reference']['STB1']},{'model':'SYN0',**core['candidates']['SYN0']}]
display(pd.DataFrame(rows).style.format({name:'{:.2%}' for name in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print(core['comparisons']); print('DECISION:',core['decision'])
assert core['decision']=='PASS','STOP: SYN0 tidak mengalahkan STB1; jangan jalankan frequency stage.'


## Stage 2 — controlled AF2 fusion
Hanya dijalankan setelah core PASS. SYN1 dan SYN2 memiliki parameter/state schema yang sama.

In [ ]:
command=base+['--stage','frequency']
print('MENJALANKAN FREQUENCY:',' '.join(command),flush=True)
subprocess.run(command,cwd=REPO,check=True)


In [ ]:
FINAL=OUTPUT/'val_reports/agsf_frequency_seed42_decision.json'
final=json.loads(FINAL.read_text())
rows=[{'model':'STB1',**final['reference']['STB1']}]+[{'model':name,**value} for name,value in final['candidates'].items()]
display(pd.DataFrame(rows).style.format({name:'{:.2%}' for name in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print('COMPARISONS:',json.dumps(final['comparisons'],indent=2))
print('DECISION:',final['decision']); print('NEXT:',final['next_action'])
print('SUMMARY:',FINAL)
print('Test tetap terkunci. Kirim tabel dan keputusan; jangan menjalankan seed lain.')
